# Домашняя работа 1 — доли исходов HMDA

## Подготовка

In [10]:
from pathlib import Path
import sys
import pandas as pd
print(sys.version)
print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Рабочая папка:", Path.cwd())
DATA = Path("../data")
assert DATA.is_dir(), "Рядом с папкой ноутбука должна находиться data/"

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Python: C:\Users\Кирилл\Desktop\анализ данных\data-analysis-course\.venv\Scripts\python.exe
pandas: 2.2.2
Рабочая папка: C:\Users\Кирилл\Desktop\анализ данных\data-analysis-course\lessons\01-introduction\work


## Основной набор decisions.csv

In [2]:
home_decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
home_outcomes = home_decisions["action_taken"]
home_n = len(home_decisions)
assert home_n > 0
assert home_outcomes.isna().sum() == 0
assert home_outcomes.isin([1, 2, 3]).sum() == home_n
print(home_decisions.shape)
home_outcomes.value_counts(dropna=False)

(6830, 7)


action_taken
1    5956
3     691
2     183
Name: count, dtype: int64

## H1. Доля отказов

In [3]:
home_denied_mask = home_outcomes.eq(3)         
home_denied = int(home_denied_mask.sum())      
home_denial_rate = home_denied / home_n         

print("Условие: action_taken == 3 (отказ)")
print("Числитель (отказы):", home_denied)
print("Знаменатель (все записи home_n):", home_n)
print("Доля:", home_denial_rate)
print(f"Процент: {home_denial_rate * 100:.2f}%")

home_approved = int(home_outcomes.isin([1, 2]).sum())   
print("Одобрения (коды 1 и 2):", home_approved)

assert home_denied + home_approved == home_n
assert 0 <= home_denial_rate <= 1

Условие: action_taken == 3 (отказ)
Числитель (отказы): 691
Знаменатель (все записи home_n): 6830
Доля: 0.10117130307467057
Процент: 10.12%
Одобрения (коды 1 и 2): 6139


## H2. Одобрение без принятия предложения

In [6]:
home_not_accepted = int(home_outcomes.eq(2).sum())
home_not_accepted_rate = home_not_accepted / home_n
print("Условие: action_taken == 2")
print("Числитель:", home_not_accepted)
print("Знаменатель:", home_n)
print(f"Процент: {home_not_accepted_rate * 100:.2f}%")
home_issued = int(home_outcomes.eq(1).sum())
print("Выдачи (код 1):", home_issued)
assert home_approved - home_issued == home_not_accepted
home_approved_rate = home_approved / home_n
home_issued_rate = home_issued / home_n
diff_pp = (home_approved_rate - home_issued_rate) * 100
print(f"Одобрения: {home_approved_rate * 100:.2f}%, выдачи: {home_issued_rate * 100:.2f}%")
print(f"Разность: {diff_pp:.2f} п.п.")
print(f"Доля исхода 2: {home_not_accepted_rate * 100:.2f}%")

Условие: action_taken == 2
Числитель: 183
Знаменатель: 6830
Процент: 2.68%
Выдачи (код 1): 5956
Одобрения: 89.88%, выдачи: 87.20%
Разность: 2.68 п.п.
Доля исхода 2: 2.68%


## H3. Объяснение инструментов

In [ ]:
Series и DataFrame. DataFrame - вся таблица, Series - один её столбец. У меня home_decisions - DataFrame, а home_outcomes = home_decisions["action_taken"] - Series. nrows и head. nrows ограничивает, сколько строк прочитает read_csv. head только показывает первые строки уже загруженной таблицы. Мне нужны все home_n записей, поэтому nrows я не использую, иначе знаменатель был бы неверным. Метод и атрибут. Метод вызывается со скобками и что-то делает, как home_outcomes.eq(3). Атрибут - готовое свойство без скобок, как home_decisions.shape. 

## Дополнительная часть

## Исходный снимок для H+1

In [4]:
raw = pd.read_csv(
    DATA / "hmda_2023_dc_raw.csv",
    dtype={"lei": "string", "county_code": "string"},
)
raw_outcomes = raw["action_taken"]
raw_n = len(raw)
assert raw_n > 0
print(raw.shape)
raw_outcomes.value_counts(dropna=False)

(8575, 99)


action_taken
1    5956
4    1585
3     691
2     183
5     160
Name: count, dtype: int64

## H+1. Доли выдач в двух наборах

In [7]:
raw_issued = int(raw_outcomes.eq(1).sum())
raw_issued_rate = raw_issued / raw_n
print("raw: числитель", raw_issued, "знаменатель", raw_n, f"-> {raw_issued_rate * 100:.2f}%")
home_issued_rate = home_issued / home_n
print("home_decisions: числитель", home_issued, "знаменатель", home_n, f"-> {home_issued_rate * 100:.2f}%")
assert raw_issued == home_issued

raw: числитель 5956 знаменатель 8575 -> 69.46%
home_decisions: числитель 5956 знаменатель 6830 -> 87.20%


## Текстовые коды для H+2

In [5]:
text_example = pd.read_csv(
    DATA / "reading_example.csv", dtype={"action_taken": "string"}
)
text_example["action_taken"]

0    1
1    1
2    2
3    2
4    3
5    3
Name: action_taken, dtype: string

## H+2. Сравнение текста с числом и текстом

In [8]:
as_number = text_example["action_taken"].eq(1)
as_text = text_example["action_taken"].eq("1")
print(as_number)
print(as_text)
print("eq(1):  True =", int(as_number.sum()))
print('eq("1"): True =', int(as_text.sum()))

0    False
1    False
2    False
3    False
4    False
5    False
Name: action_taken, dtype: boolean
0     True
1     True
2    False
3    False
4    False
5    False
Name: action_taken, dtype: boolean
eq(1):  True = 0
eq("1"): True = 2
